In [3]:
import warnings
warnings.filterwarnings("ignore")

### Data Preprocessing:

The text dataset is processed and transformed, to use it for model training.
Steps:
- Loading Dataset
- Initialising a tokenizer object for the pretrained model of BERT Transformer
- The transformations on dataset involve
  - Encoding the words with their respective word 'id' from the BERT Model
  - The sentences are padded to make all the sentences of same length in our case the max_length = 200
  - Respective attention vectors are created for each sentence with value '1' for word and value '0' for padded words.
- This transformed requires to have a type of tensorflow tensor data. So we will transform the data into Tensorflow tensor data format.

In [4]:
from datasets import load_dataset 

dataset = load_dataset("ag_news")
split = dataset['train'].train_test_split(test_size=0.1, seed=42)
train_ds = split["train"]
val_ds = split['test']
test_ds = dataset['test']

In [5]:
from transformers import AutoTokenizer

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [6]:
def encode_batch(batch):
    return tokenizer(
        batch['text'],
        max_length = 200,
        truncation = True,
        padding = "max_length"
    )

train_enc = train_ds.map(encode_batch, batched=True)
test_enc = test_ds.map(encode_batch, batched=True)
val_enc = val_ds.map(encode_batch, batched=True)


train_enc = train_enc.rename_column("label", "labels")
test_enc = test_enc.rename_column("label", "labels")
val_enc = val_enc.rename_column("label", "labels")


train_enc.set_format(type = 'tensorflow', columns = ['input_ids','attention_mask', 'labels'])
test_enc.set_format(type = 'tensorflow', columns = ['input_ids','attention_mask', 'labels'])
val_enc.set_format(type = 'tensorflow', columns = ['input_ids','attention_mask', 'labels'])

In [7]:
import tensorflow as tf

tf_train = train_enc.to_tf_dataset(
    columns=["input_ids", "attention_mask"],
    label_cols=["labels"],
    shuffle=False,
    batch_size=64,
)


tf_val = val_enc.to_tf_dataset(
    columns=["input_ids","attention_mask"],
    label_cols=["labels"],
    shuffle=False,
    batch_size=64,
)

tf_test = test_enc.to_tf_dataset(
    columns=["input_ids", "attention_mask"],
    label_cols=["labels"],
    shuffle=False,
    batch_size=64,
)

### Model Initialization

For the text classification task, we employ the BERT Base Uncased model, which is a pretrained Transformer-based encoder.
BERT is a bidirectional autoencoder architecture trained on large-scale corpora using two objectives:

- Masked Language Modeling (MLM) — learns deep contextual word representations
- Next Sentence Prediction (NSP) — learns relationships between sentences

Because of this pretraining, BERT produces rich semantic embeddings for every token and for the entire sentence through the [CLS] embedding.

#### How BERT is used in our classification task

For sequence (text) classification, we follow the standard fine-tuning setup:

- The input text is tokenized into word-piece tokens and converted to integer IDs.

- These token IDs and their attention masks are passed into BERT’s encoder stack.

- BERT produces a contextual embedding for every token and a special [CLS] embedding, which summarizes the meaning of the entire input sequence.

On top of this [CLS] embedding, we place a fully connected classification layer:

$$
\mathbf{h}_{\text{CLS}} \in \mathbb{R}^{768} \;\longrightarrow\; \text{Dense}(4)
$$


where 4 corresponds to the four AG News categories.

This classification layer is randomly initialized, and only its weights are trained from scratch, while BERT’s encoder weights are fine-tuned during training to adapt the representations to our dataset.

#### What the model learns

- BERT provides context-aware embeddings that capture syntax, semantics, and long-range dependencies.

- The classification head learns to map these embeddings to a 4-class probability distribution using a softmax activation.

- Training optimizes the cross-entropy loss between predicted and true labels.

In [20]:
from transformers import AutoTokenizer, TFBertModel

model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

bert = TFBertModel.from_pretrained(
    model_name,
    from_pt=True  # IMPORTANT: loads PyTorch weights into TF
)

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.bias', 'cls.predictions.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already

In [21]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.models import Model

max_len = 200

input_ids = Input(shape=(max_len,), dtype=tf.int32, name="input_ids")
attention_mask = Input(shape=(max_len,), dtype=tf.int32, name="attention_mask")

In [22]:
# Add output shape

bert_output = tf.keras.layers.Lambda(
    lambda x: bert(x["input_ids"], attention_mask=x["attention_mask"])[0],
    output_shape=(max_len, 768)
)({"input_ids": input_ids, "attention_mask": attention_mask})

cls_embedding = bert_output[:, 0, :]  # shape = (batch, 768)


x = tf.keras.layers.Dropout(0.2)(cls_embedding)
x = tf.keras.layers.Dense(128, activation="relu")(x)
output = tf.keras.layers.Dense(4, activation="softmax")(x)

model = tf.keras.Model(inputs=[input_ids, attention_mask], outputs=output)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ attention_mask      │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_ids           │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_2 (Lambda)   │ (None, 200, 768)  │          0 │ attention_mask[0… │
│                     │                   │            │ input_ids[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_2          │ (None, 768)       │          0 │ lambda_2[0][0]    │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 768)       │          0 │ get_item_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 128)       │     98,432 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 4)         │        516 │ dense_4[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 98,948 (386.52 KB)

 Trainable params: 98,948 (386.52 KB)

 Non-trainable params: 0 (0.00 B)

In [24]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [25]:
%%time 
import time
cpu_start_time = time.process_time()
with tf.device("/GPU:0"):
    history = model.fit(
        tf_train,
        validation_data = tf_val,
        epochs = 10
    )
cpu_end_time = time.process_time()
cpu_train_time = cpu_end_time - cpu_start_time
print(cpu_train_time)

Epoch 1/10
1687/1688 ━━━━━━━━━━━━━━━━━━━━ 0s 847ms/step - accuracy: 0.5098 - loss: 1.1497

W0000 00:00:1764469058.804538     128 assert_op.cc:38] Ignoring Assert operator functional_2_1/lambda_2_1/tf_bert_model_2/bert/embeddings/assert_less/Assert/Assert


1688/1688 ━━━━━━━━━━━━━━━━━━━━ 0s 849ms/step - accuracy: 0.5099 - loss: 1.1495

W0000 00:00:1764469066.382980     125 assert_op.cc:38] Ignoring Assert operator functional_2_1/lambda_2_1/tf_bert_model_2/bert/embeddings/assert_less/Assert/Assert
W0000 00:00:1764469227.605447     128 assert_op.cc:38] Ignoring Assert operator functional_2_1/lambda_2_1/tf_bert_model_2/bert/embeddings/assert_less/Assert/Assert


1688/1688 ━━━━━━━━━━━━━━━━━━━━ 1599s 948ms/step - accuracy: 0.5100 - loss: 1.1494 - val_accuracy: 0.8293 - val_loss: 0.5521
Epoch 2/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 1587s 940ms/step - accuracy: 0.8215 - loss: 0.5407 - val_accuracy: 0.8587 - val_loss: 0.4249
Epoch 3/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 1589s 941ms/step - accuracy: 0.8534 - loss: 0.4344 - val_accuracy: 0.8717 - val_loss: 0.3820
Epoch 4/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 1588s 941ms/step - accuracy: 0.8663 - loss: 0.3918 - val_accuracy: 0.8779 - val_loss: 0.3598
Epoch 5/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 1586s 940ms/step - accuracy: 0.8721 - loss: 0.3743 - val_accuracy: 0.8810 - val_loss: 0.3465
Epoch 6/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 1588s 941ms/step - accuracy: 0.8749 - loss: 0.3596 - val_accuracy: 0.8844 - val_loss: 0.3368
Epoch 7/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 1587s 940ms/step - accuracy: 0.8779 - loss: 0.3499 - val_accuracy: 0.8865 - val_loss: 0.3298
Epoch 8/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 1588s 941ms/step - accur

#### Model Evaluation:

We will check the model on training and test dataset and collect various model metrics, They are:
- Train Dataset Metrics
- Test Dataset Metrics
- Metrics used:
  - Accuracy
  - F1 Score
- Inference Time of Test Dataset
- FLOP's for each Sample Inference

In [40]:
model.save("/kaggle/working/my_model.keras")
tokenizer.save_pretrained('tokenizer')


('tokenizer/tokenizer_config.json',
 'tokenizer/special_tokens_map.json',
 'tokenizer/vocab.txt',
 'tokenizer/added_tokens.json',
 'tokenizer/tokenizer.json')

In [10]:
from tensorflow.keras.models import load_model

model = load_model("codes/my_model.keras")
tokenizer = AutoTokenizer.from_pretrained("codes/tokenizer")

OSError: No file or directory found at codes/my_model.keras/

In [33]:
%%time
import time
import numpy as np


# Start CPU timer
cpu_start_time = time.process_time()
start_wall = time.time()

# Run prediction
y_pred_probs = model.predict(tf_test
)

y_test_pred = np.argmax(y_pred_probs, axis=1)

# Stop timers
cpu_end_time = time.process_time()
end_wall = time.time()

cpu_time_inf = cpu_end_time - cpu_start_time
wall_time_inf = end_wall - start_wall

print("CPU inference time:", cpu_time_inf, "seconds")
print("Wall-clock inference time:", wall_time_inf, "seconds")

119/119 ━━━━━━━━━━━━━━━━━━━━ 101s 852ms/step
CPU inference time: 101.56264206200012 seconds
Wall-clock inference time: 101.43941903114319 seconds
CPU times: user 1min 41s, sys: 0 ns, total: 1min 41s
Wall time: 1min 41s


In [34]:
y_train_probs = model.predict(
        tf_train
    # {
    #     "input_ids": train_enc["input_ids"],
    #     "attention_mask": train_enc["attention_mask"]
    # },
    # batch_size=32
)

y_train_pred = np.argmax(y_train_probs, axis=1)

1687/1688 ━━━━━━━━━━━━━━━━━━━━ 0s 848ms/step

W0000 00:00:1764486565.963604     126 assert_op.cc:38] Ignoring Assert operator functional_2_1/lambda_2_1/tf_bert_model_2/bert/embeddings/assert_less/Assert/Assert


1688/1688 ━━━━━━━━━━━━━━━━━━━━ 1433s 849ms/step


In [49]:
y_train = np.array(train_enc['labels'])

In [50]:
from sklearn.metrics import accuracy_score,f1_score, classification_report
print("Training Accuracy")
train_accuracy = accuracy_score(y_train, y_train_pred)
train_f1 = f1_score(y_train, y_train_pred, average='weighted', zero_division=0)
train_report = classification_report(y_train, y_train_pred)
print('\n Training Accuracy',train_accuracy)
print('\n Training F1 Score',train_f1)
print('\n Training Classification Report',train_report)

Training Accuracy

 Training Accuracy 0.24912037037037038

 Training F1 Score 0.24907870301743473

 Training Classification Report               precision    recall  f1-score   support

           0       0.25      0.24      0.24     26991
           1       0.25      0.25      0.25     26966
           2       0.25      0.24      0.25     27100
           3       0.25      0.26      0.25     26943

    accuracy                           0.25    108000
   macro avg       0.25      0.25      0.25    108000
weighted avg       0.25      0.25      0.25    108000



In [42]:
y_test = test_enc['labels']

In [44]:
print("Test Accuracy")
test_accuracy = accuracy_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred, average='weighted', zero_division=0)
test_report = classification_report(y_test,y_test_pred)
print('\n Test Accuracy',test_accuracy)
print('\n Test F1 Score',test_f1)
print('\n Test Classification Report',test_report)

Test Accuracy

 Test Accuracy 0.8889473684210526

 Test F1 Score 0.8887908168034738

 Test Classification Report               precision    recall  f1-score   support

           0       0.90      0.89      0.89      1900
           1       0.96      0.97      0.96      1900
           2       0.86      0.83      0.84      1900
           3       0.84      0.87      0.86      1900

    accuracy                           0.89      7600
   macro avg       0.89      0.89      0.89      7600
weighted avg       0.89      0.89      0.89      7600



##### FLOP's Calculation: 
A Single BERT Layer includes:

- Q, K, V projections → 3 matrix multiplications

- Attention softmax

- Attention × V

- Feed-forward network (two dense layers)

- Layer norm, dropout, residuals

A 12-layer BERT-base model ≈ 22Billion FLOPs per sequence of length 200.

In [ ]:
flops = 22_000_000_000

In [ ]:
results = {}
results[('BERT Transformer','MLP Model')] = {
                                    'Train':
                                            {'accuracy': train_accuracy,
                                            'f1_score': train_f1,
                                            'report': train_report,
                                            'training_time': cpu_train_time},
                                    'Test':
                                            {'accuracy': test_accuracy,
                                            'f1_score': test_f1,
                                            'report': test_report,
                                            'inference_time':cpu_time_inf,
                                            "flops/sample": flops}}

## Tabulate:

In [ ]:
# create DataFrame
import pandas as pd

# flatten nested dict into a list of rows
rows = []

for (vectorizer, model), metrics in results.items():
    row = {
        "Vectorizer": vectorizer,
        "Model": model,
        "Train_Accuracy": metrics["Train"]["accuracy"],
        "Train_F1": metrics["Train"]["f1_score"],
        "Test_Accuracy": metrics["Test"]["accuracy"],
        "Test_F1": metrics["Test"]["f1_score"],
        "Inference_Time": metrics["Test"].get("inference_time", None),
        "FLOPs_per_Sample": metrics["Test"].get("flops/sample", None),
        "Training_Time": metrics['Train'].get('training_time',None)
    }
    rows.append(row)

# create DataFrame
df_results = pd.DataFrame(rows)

# optional: set a clean display order
df_results = df_results[
    ["Vectorizer", "Model", "Train_Accuracy", "Train_F1",
     "Test_Accuracy", "Test_F1", "Inference_Time", "FLOPs_per_Sample","Training_Time"]
]

print(df_results)

In [ ]:
df_results.to_csv('/kaggle/working/w2vec_emb.csv')

In [ ]:
df_results

In [53]:
tf_train.label_cols

AttributeError: '_PrefetchDataset' object has no attribute 'label_cols'

In [54]:
model.evaluate(tf_train)

1688/1688 ━━━━━━━━━━━━━━━━━━━━ 1433s 849ms/step - accuracy: 0.8950 - loss: 0.3065


[0.30610090494155884, 0.8945833444595337]